# Train the "Hey Huncho" wake word — v2 (leaner + Drive-safe)

Trains a custom openWakeWord-compatible ONNX model with [livekit-wakeword](https://github.com/livekit/livekit-wakeword). Fully synthetic data (local Piper TTS), no API keys.

**v2 changes:** ~4x faster than v1 (15k steps vs 50k), adversarial negatives for "huncho", and the output lands in your **Google Drive** so a disconnect can't lose it.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all** (approve the Drive access prompt when it appears).

**Expect roughly 45–90 min.** The training cell prints step counters — climbing numbers = working, not stuck. Keep the tab open.

In [ ]:
# 1) Mount Google Drive (output goes here so nothing is lost on disconnect)
from google.colab import drive
drive.mount('/content/drive')
import pathlib
OUT = '/content/drive/MyDrive/huncho_wakeword'
pathlib.Path(OUT).mkdir(parents=True, exist_ok=True)
print('output dir:', OUT)
!nvidia-smi -L  # should print a GPU; if not, fix Runtime type before continuing

In [ ]:
# 2) System deps + trainer
!apt-get -qq update && apt-get -qq install -y espeak-ng libsndfile1 ffmpeg sox > /dev/null
%pip install -q "livekit-wakeword[train,eval,export]"
print('installed OK')

In [ ]:
# 3) Training config — lean personal-use run with adversarial negatives
config = f'''\
model_name: huncho
target_phrases: ["hey huncho", "huncho"]

n_samples: 5000
n_samples_val: 750
n_background_samples: 1000
n_background_samples_val: 250
tts_batch_size: 50

# things that must NOT wake him (phonetic neighbors)
custom_negative_phrases:
  - "honcho"
  - "head honcho"
  - "hunch"
  - "hunches"
  - "poncho"
  - "hey poncho"
  - "nacho"
  - "hey nacho"
  - "hey hunter"
  - "hey honda"
  - "hey uncle"
  - "lunch oh"

data_dir: ./data
output_dir: {OUT}/output

augmentation:
  clip_duration: 2.0
  batch_size: 16
  rounds: 2
  background_paths: [./data/backgrounds]
  rir_paths: [./data/rirs]

model:
  model_type: conv_attention
  model_size: small

steps: 15000
learning_rate: 0.0001
weight_decay: 0.01
label_smoothing: 0.05
max_negative_weight: 1500
target_fp_per_hour: 0.5

batch_n_per_class:
  positive: 50
  adversarial_negative: 50
  ACAV100M_sample: 512
  background_noise: 50
'''
pathlib.Path('huncho.yaml').write_text(config)
print(config)

In [ ]:
# 4) Download assets, then run the full pipeline (synthesize -> augment -> train -> export)
#    Training prints step counters — climbing = working. ~45-90 min on T4.
!livekit-wakeword setup --config huncho.yaml
!livekit-wakeword run huncho.yaml

In [ ]:
# 5) Grab the exported ONNX (also already safe in your Drive) and download it
import glob, shutil
candidates = glob.glob(OUT + '/**/*.onnx', recursive=True)
print('found:', candidates)
src = [c for c in candidates if 'huncho' in c.lower()] or candidates
shutil.copy(src[0], 'huncho.onnx')
from google.colab import files
files.download('huncho.onnx')